# Notebook 02 — Data Cleaning & SQLite Loading

Takes the raw CSVs from Notebook 01 and:
1. Cleans and normalizes all fields
2. Tags racial equity grants via keyword matching (`src/data_cleaning.py`)
3. Loads everything into a local SQLite database

The database is rebuilt from scratch each run, so this notebook is safe to re-run.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd

from src.data_cleaning import (
    clean_grants_df,
    clean_foundations_df,
    normalize_org_name,
    get_connection,
    load_table,
)

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)
DB   = Path('../data/racial_equity_grants.sqlite')

## 1  Initialize a fresh SQLite database

We delete any existing database and recreate the schema from `sql/create_tables.sql` so
repeated runs don't duplicate rows.

In [ ]:
for p in [DB, Path(str(DB) + '-wal'), Path(str(DB) + '-shm')]:
    if p.exists():
        p.unlink()

conn = get_connection(DB)
with open('../sql/create_tables.sql') as f:
    conn.executescript(f.read())
conn.commit()
print('Database initialized:', DB)

## 2  Clean grants

**Cleaning decisions** (see `src/data_cleaning.py`):
- Normalize EINs to 9-digit zero-padded strings.
- Drop rows missing `funder_ein` or `grant_amount` (can't attribute / not dollar-weightable).
- Uppercase + de-punctuate org names for consistent matching.
- Coerce `tax_year` to integer; unparseable years become null and drop out of time series.
- `is_racial_equity` flag set by keyword regex on `grant_purpose`.
- 990-PF grants carry no recipient EIN, so `recipient_ein` is null by design.

In [ ]:
grants_raw = pd.read_csv(RAW / 'grants_raw.csv', dtype=str)
print(f"Raw grants: {len(grants_raw):,} rows")

grants = clean_grants_df(grants_raw)
grants['source'] = 'irs_990pf'
print(f"Cleaned grants: {len(grants):,} rows")
print(f"Racial equity grants: {grants['is_racial_equity'].sum():,} "
      f"({grants['is_racial_equity'].mean():.1%} of grants)")
grants.head()

In [ ]:
# Anomaly check: grant amounts
print('Grant amount summary:')
print(grants['grant_amount'].describe().apply(lambda x: f'${x:,.0f}'))

outliers = grants[grants['grant_amount'] > 500_000_000]
print(f"\nGrants over $500M (review for data errors): {len(outliers)}")

## 3  Clean foundations (funders)

In [ ]:
funders_raw = pd.read_csv(RAW / 'funders_raw.csv', dtype=str)
funders = clean_foundations_df(funders_raw)
funders = funders.sort_values('tax_year', ascending=False).drop_duplicates('ein')
funders['source'] = 'irs_990pf'
print(f"Unique foundations: {len(funders):,}")
funders.head()

## 4  Clean recipients

Recipients come from the grant records (name + city + state, no EIN). We normalize the name
and keep `ntee_code` / `total_revenue` as null columns — these would be filled by the Candid
integration, which can resolve recipient EINs.

In [ ]:
recipients_raw = pd.read_csv(RAW / 'recipients_raw.csv', dtype=str)
recipients = recipients_raw.copy()
recipients['name'] = recipients['name'].apply(normalize_org_name)
recipients = recipients.dropna(subset=['name']).drop_duplicates(['name', 'state'])
recipients['ntee_major'] = recipients['ntee_code'].str[:1].str.upper()
recipients['total_revenue'] = pd.to_numeric(recipients['total_revenue'], errors='coerce')
recipients['source'] = 'irs_990pf'
print(f"Unique recipients: {len(recipients):,}")
recipients.head()

## 5  Load into SQLite

Loaded in foreign-key order: `foundations` and `recipients` first, then `grants`. Only the
columns defined in the schema are written.

In [ ]:
FOUNDATION_COLS = ['ein', 'name', 'tax_year', 'state', 'total_assets',
                   'total_revenue', 'total_grants_paid', 'source']
RECIPIENT_COLS  = ['ein', 'name', 'ntee_code', 'ntee_major', 'state', 'city',
                   'total_revenue', 'source']
GRANT_COLS      = ['funder_ein', 'recipient_ein', 'recipient_name', 'recipient_city',
                   'recipient_state', 'grant_amount', 'tax_year', 'grant_purpose',
                   'is_racial_equity', 'source', 'object_id']

def aligned(df, cols):
    """Return df with exactly `cols` (missing columns added as None)."""
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = None
    return out[cols]

# Save cleaned CSVs for the record
grants.to_csv(PROC / 'grants_clean.csv', index=False)
funders.to_csv(PROC / 'foundations_clean.csv', index=False)
recipients.to_csv(PROC / 'recipients_clean.csv', index=False)

load_table(aligned(funders, FOUNDATION_COLS), 'foundations', conn)
load_table(aligned(recipients, RECIPIENT_COLS), 'recipients', conn)
load_table(aligned(grants, GRANT_COLS), 'grants', conn)
conn.commit()

## 6  Validation

In [ ]:
checks = {
    'Total grants': 'SELECT COUNT(*) FROM grants',
    'Racial equity grants': 'SELECT COUNT(*) FROM grants WHERE is_racial_equity=1',
    'Unique funders': 'SELECT COUNT(DISTINCT funder_ein) FROM grants',
    'Total RE dollars ($M)': 'SELECT ROUND(SUM(grant_amount)/1e6,2) FROM grants WHERE is_racial_equity=1',
    'Foundations in DB': 'SELECT COUNT(*) FROM foundations',
    'Recipients in DB': 'SELECT COUNT(*) FROM recipients',
}
for label, sql in checks.items():
    print(f"{label:28s} {conn.execute(sql).fetchone()[0]}")

## Summary
Cleaned data is in `data/processed/` and loaded into `data/racial_equity_grants.sqlite`.

Proceed to **Notebook 03** for exploratory analysis.